# Titanic Dataset: Data Cleaning & Preprocessing

This notebook walks through cleaning the classic [Kaggle Titanic dataset](https://www.kaggle.com/competitions/titanic/data) (passenger records from the 1912 sinking).

The goal here is **cleaning**, not modeling: we'll look for missing values, duplicates, wrong data types, and inconsistent values, and fix each one step by step before doing a little light preprocessing at the end.

In [2]:
import pandas as pd

df = pd.read_csv('titanic.csv')
df.head(15)

FileNotFoundError: [Errno 2] No such file or directory: 'titanic.csv'

We can show the unique values in a column using this function:

In [ ]:
df['Pclass'].unique()

We can check why the Age was stored as a decimal number:

In [ ]:
df['Age'].unique()

In [ ]:
# Merging another set of data (extra_passenger_data)
# merged_data = pd.merge(df, extra_passenger_data, on='PassengerId', how='left')

## 1. First look at the data

Before cleaning anything, get a feel for the shape of the data, the column types, and the summary statistics.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Checking for missing values

`df.info()` above already hints at missing data (some columns have fewer non-null entries than the total row count), but `isnull().sum()` gives us a clear column-by-column count.

In [ ]:
df.isnull().sum()

Three columns have missing values, and they're each missing for a different reason:

- **`Age`** — 177 missing (~20% of passengers). Age just wasn't recorded for everyone.
- **`Cabin`** — 687 missing (~77% of passengers). Most people, especially in lower classes, simply weren't assigned a recorded cabin.
- **`Embarked`** — only 2 missing. Almost certainly a couple of records where the port just wasn't logged.

Each of these needs a different fix — we'll handle them one at a time below.

## 3. Checking for duplicate rows

It's also worth checking whether any passenger got recorded twice, either as an exact duplicate row or a duplicate `PassengerId`.

In [ ]:
print('df.duplicated:', df.duplicated())
print('Fully duplicated rows:', df.duplicated().sum())
print('Duplicate PassengerId values:', df['PassengerId'].duplicated().sum())

No duplicates here — nothing to fix in this section, but it's a check you should always run rather than assume.

## 4. Cleaning missing values

### 4a. `Embarked` — fill with the most common value

Only 2 rows are missing `Embarked`. With so few missing, the safest and simplest fix is to fill them with the most frequent port (the *mode*), rather than dropping the rows and losing data.

In [ ]:
df['Embarked'].value_counts(dropna=False)

In [ ]:
most_common_port = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(most_common_port)

df['Embarked'].isnull().sum()

### 4b. `Age` — fill with the median (grouped by passenger class)

Age is missing for ~20% of passengers, so it's too much data to just drop. A single overall median would work, but age tends to vary a lot by ticket class (`Pclass`) — first class passengers were generally older than third class passengers. So instead of one number for everyone, we'll fill each missing age with the **median age for that passenger's class**.

In [ ]:
df.groupby('Pclass')['Age'].median()

In [ ]:
df['Age'] = df['Age'].fillna(df.groupby('Pclass')['Age'].transform('median'))

df['Age'].isnull().sum()

**How this line works:**

1. `df.groupby('Pclass')['Age']` splits the rows into groups by passenger class, then selects the `Age` column within each group.
2. `.transform('median')` computes the median age *within each group*, then broadcasts that value back out to every row in the group — unlike plain `.median()`, which would collapse each group down to a single row, `transform` returns a result the same length as the original `df` (891 rows), aligned row-for-row.
3. `df['Age'].fillna(...)` uses that per-row Series to fill in only the missing ages — each `NaN` gets replaced by the median for *that passenger's own class*, while existing ages are left untouched.
4. `df['Age'] = ...` assigns the filled column back, overwriting `Age` in place.

### 4c. `Cabin` — extract what we can, then drop it

`Cabin` is missing for ~77% of passengers, which is far too much to safely impute a specific cabin number. But rather than throwing the whole column away, we can pull out one useful piece of information: the **deck letter** (the first character of the cabin code, e.g. `"C85"` → `"C"`). Passengers with no recorded cabin get labeled `"Unknown"` instead of being dropped.

In [3]:
df['Cabin'].unique()

NameError: name 'df' is not defined

In [ ]:
df['Deck'] = df['Cabin'].str[0]
df['Deck'] = df['Deck'].fillna('Unknown')

df['Deck'].value_counts(dropna=False)

In [ ]:
df.head()

In [ ]:
# Now that we've kept the useful part (Deck), the original Cabin column can go
df = df.drop(columns=['Cabin'])
df.columns

Let's confirm all the missing values are gone.

In [ ]:
df.isnull().sum()

## 5. Fixing data types

Several columns are stored as numbers or generic objects but are really **categories** — a fixed, small set of labels rather than something you'd do arithmetic on. `Survived` (0/1), `Pclass` (1/2/3), `Sex`, `Embarked`, and our new `Deck` column all fit this description. Converting them to pandas' `category` dtype makes that explicit and uses less memory.

In [ ]:
categorical_columns = ['Survived', 'Pclass', 'Sex', 'Embarked', 'Deck']
df[categorical_columns] = df[categorical_columns].astype('category')

df.dtypes

## 6. Checking for inconsistent or suspicious values

Missing values aren't the only kind of "dirty" data — a column can be fully populated and still contain values that look wrong. Let's sanity-check the two main numeric columns, `Fare` and `Age`.

In [ ]:
df['Fare'].describe()

In [ ]:
zero_fare = df[df['Fare'] == 0]
print(f"Passengers with Fare == 0: {len(zero_fare)}")
zero_fare[['Name', 'Pclass', 'Fare']]

15 passengers paid a fare of exactly 0. That's not necessarily an *error* — historical records show a number of these were crew members or employees of the shipping line traveling for free — but it's worth flagging rather than silently trusting. We'll leave these values as-is rather than guessing a replacement, since we don't have strong evidence they're wrong.

Now a quick sanity check on `Age`: does the range make sense?

In [ ]:
print('Youngest passenger:', df['Age'].min())
print('Oldest passenger:', df['Age'].max())

0.42 years old (about 5 months) up to 80 — both are plausible for a passenger ship, so nothing to fix here. It's a reminder that "checking for outliers" doesn't mean removing every unusual value, just the ones that are actually impossible or nonsensical (e.g. a negative age, or a 300-year-old passenger).

## 7. A little light preprocessing

The data is now clean. Preprocessing goes a step further — reshaping clean data so it's more useful for analysis. Two small, common examples:

**`FamilySize`** — `SibSp` (siblings/spouses aboard) and `Parch` (parents/children aboard) are more useful combined into one number, plus 1 for the passenger themselves.

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

df[['SibSp', 'Parch', 'FamilySize']].head()

**`Sex_encoded`** — `Sex` is text (`"male"`/`"female"`). Many analysis and plotting operations are easier with numbers, so we map it to 0/1 in a new column (keeping the original `Sex` column too, since it's more readable).

In [ ]:
df['Sex_encoded'] = df['Sex'].map({'male': 0, 'female': 1})

df[['Sex', 'Sex_encoded']].head()

## 8. Dropping columns that aren't useful for analysis

`Name` and `Ticket` are mostly unique, free-text identifiers — every passenger has a different one, so they don't help us summarize or group the data. We'll drop them (an analyst who needed the name for lookup purposes could keep it, but for our purposes it's just noise).

In [ ]:
df = df.drop(columns=['Name', 'Ticket'])
df.columns

## 9. Detecting outliers with the IQR method

An **outlier** is a value that sits unusually far from the rest of the data in its column. A common, distribution-free way to flag them is the **IQR (interquartile range) method**:

- **Q1** = 25th percentile, **Q3** = 75th percentile
- **IQR** = Q3 − Q1
- **lower bound** = Q1 − 1.5 × IQR, **upper bound** = Q3 + 1.5 × IQR
- any value outside `[lower bound, upper bound]` is flagged as an outlier

This only makes sense for continuous numeric columns with a meaningful spread. `Age` and `Fare` fit that description. `SibSp`, `Parch`, and `FamilySize` are small counts that are mostly 0 or 1 — running IQR on them would flag most nonzero rows as "outliers," which isn't a meaningful signal, so we'll skip those.

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ['Age', 'Fare']:
    lower, upper = iqr_bounds(df[col])
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: bounds=({lower:.2f}, {upper:.2f}), outliers={n_outliers} ({n_outliers / len(df):.1%})")

`Age`'s lower bound comes out negative, which is below the minimum possible age — so in practice only the *upper* cutoff matters for `Age` (unusually old passengers). `Fare` has a real, meaningful lower bound too, but since `Fare` can't go below 0 either, its lower cutoff being negative means only the upper cutoff is meaningful there as well. Let's visualize where those upper cutoffs fall.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col in zip(axes, ['Age', 'Fare']):
    lower, upper = iqr_bounds(df[col])
    ax.hist(df[col], bins=30, color='#4C72B0', edgecolor='white')
    if lower > df[col].min():
        ax.axvline(lower, color='crimson', linestyle='--', label=f'lower cutoff ({lower:.1f})')
    ax.axvline(upper, color='crimson', linestyle='--', label=f'upper cutoff ({upper:.1f})')
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.legend(fontsize=8) 

plt.tight_layout()
plt.show()

`Fare` has a long right tail — about 13% of passengers land above the IQR cutoff, including the well-known $512 fare. `Age` has a much smaller tail, with 26 passengers (mostly elderly) flagged above ~59.5.

**Should we actually remove them?** Not here. IQR flags statistically unusual values, but "unusual" isn't the same as "wrong." An 80-year-old passenger and a first-class fare of $512 are real, legitimate data points — not typos or sensor glitches — so deleting them would throw away true information rather than fix an error, the same judgment call we made with the `Fare == 0` rows earlier. Outlier *removal* is more appropriate when you have reason to believe the extreme values are errors, or when a specific downstream method (e.g. linear regression, k-means, anything distance-based) is known to be sensitive to them.

For reference, here's what removal *would* look like — applied to a separate copy so it doesn't affect `df` or the sections below:

In [ ]:
fare_lower, fare_upper = iqr_bounds(df['Fare'])
df_no_fare_outliers = df[(df['Fare'] >= fare_lower) & (df['Fare'] <= fare_upper)]

print(f"Rows before: {len(df)}")
print(f"Rows after removing Fare outliers: {len(df_no_fare_outliers)}")

## 10. Final check

One last look to confirm the dataset is clean: no missing values, sensible dtypes, and the new columns in place.

In [ ]:
df.info()

In [ ]:
print('Total missing values remaining:', df.isnull().sum().sum())
df.head()

## 11. Save the cleaned data

Now that it's cleaned and lightly preprocessed, save it to a new file so later analysis can start from here instead of re-running the cleaning steps.

In [ ]:
df.to_csv('titanic_cleaned.csv', index=False)

## Summary — was there much to clean?

Yes, a moderate amount — this dataset is a common teaching choice precisely *because* it isn't spotless:

| Issue | Found | Fix |
|---|---|---|
| Missing `Age` | 177 rows (~20%) | Filled with median age per `Pclass` |
| Missing `Cabin` | 687 rows (~77%) | Extracted `Deck` letter, filled rest as `"Unknown"`, dropped original column |
| Missing `Embarked` | 2 rows | Filled with the mode (`"S"`) |
| Duplicate rows | 0 | None found |
| Wrong data types | 5 columns | Converted to `category` dtype |
| Suspicious values | 15 rows with `Fare == 0` | Flagged, not altered (plausible, not clearly wrong) |
| Outliers | None | Age range (0.42–80) is plausible |

So while there were **no duplicates and no impossible values**, the missing data was significant — especially `Cabin`, which was missing for more than three-quarters of passengers. That's what made this a useful dataset to practice on: it needed real decisions (impute vs. drop vs. flag), not just a single `dropna()` call.

## 12. A note on data normalization

Normalization rescales numeric columns onto a common range so that no single feature dominates just because it happens to use bigger numbers. For example, `Fare` ranges from 0 to over 500, while `Age` ranges from about 0 to 80 — many models and distance-based comparisons would let `Fare` overpower `Age` purely due to scale, not because it's actually more important.

A common technique is **min-max scaling**, which squeezes every value into the range [0, 1]:

$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

This is a *preprocessing* step (it doesn't fix any data quality problem — the data is already clean), so we're just illustrating it here rather than applying it to the saved dataset.

In [ ]:
fare_min, fare_max = df['Fare'].min(), df['Fare'].max()
fare_normalized = (df['Fare'] - fare_min) / (fare_max - fare_min)

fare_normalized.describe()

## 13. Does min-max scaling preserve the shape of the distribution?

Yes. The formula `(x - min) / (max - min)` is a **linear (affine) transformation** — every value is shifted and stretched by the *same* factor, so relative distances between points stay proportional to the originals.

That means skewness, multimodality, and the overall silhouette of the histogram are unchanged by min-max scaling — `Fare`'s right-skewed distribution is still just as right-skewed after scaling, only now compressed into `[0, 1]` instead of `[0, 512]`. Only scale-dependent statistics (mean, standard deviation, min, max) change.

This is different from a **nonlinear** transform like `log`, which compresses large values more than small ones and can genuinely change the shape (e.g. pulling in a long right tail). So min-max scaling is a purely cosmetic rescale of the axis — it doesn't fix skew, and it isn't meant to.

## 14. Data reduction and PCA

**Data reduction** means representing data with fewer columns while keeping as much of the meaningful information (variance) as possible. It's useful when columns are redundant or correlated, when you want to visualize high-dimensional data in 2D, or just to cut down noise before further analysis.

**PCA (Principal Component Analysis)** is the most common technique for this. It finds new axes — each one a weighted combination of the original numeric columns — ordered so the first axis (the *first principal component*) captures as much of the total variance as possible, the second captures the next most (while staying uncorrelated with the first), and so on. Keeping only the first component or two often retains most of the signal while dropping dimensions entirely.

PCA is sensitive to scale (it looks at variance, and a column with a bigger numeric range will dominate), so it's typically applied to *standardized* data — another reason scaling and PCA are usually discussed together.